In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# ── 1. LOAD ──────────────────────────────────────────
train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

print(f"Train: {train.shape}, Test: {test.shape}")

# ── 2. KEEP OUTLIERS — just use log instead ──────────
# Don't remove any houses this time!
# Log handles skew naturally
y_train = np.log1p(train['SalePrice'])
print(f"Target range: {y_train.min():.2f} - {y_train.max():.2f}")

Train: (1460, 81), Test: (1459, 80)
Target range: 10.46 - 13.53


In [3]:
# ── 3. FEATURE ENGINEERING ───────────────────────────
for df in [train, test]:
    # Total square footage 
    df['TotalSF'] = (df['TotalBsmtSF'].fillna(0) + 
                     df['1stFlrSF'] + 
                     df['2ndFlrSF'])
    
    # Age related
    df['HouseAge']    = df['YrSold'] - df['YearBuilt']
    df['RemodAge']    = df['YrSold'] - df['YearRemodAdd']
    df['Remodeled']   = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)
    
    # Total bathrooms
    df['TotalBath'] = (df['FullBath'] + 
                       df['BsmtFullBath'].fillna(0) +
                       0.5 * df['HalfBath'] + 
                       0.5 * df['BsmtHalfBath'].fillna(0))
    
    # Porch total area
    df['TotalPorch'] = (df['OpenPorchSF'] + 
                        df['EnclosedPorch'] + 
                        df['3SsnPorch'] + 
                        df['ScreenPorch'])
    
    # Has features flags
    df['HasPool']     = (df['PoolArea'] > 0).astype(int)
    df['HasGarage']   = (df['GarageArea'].fillna(0) > 0).astype(int)
    df['HasBasement'] = (df['TotalBsmtSF'].fillna(0) > 0).astype(int)
    df['HasFireplace']= (df['Fireplaces'] > 0).astype(int)
    
    # ── NEW: Interaction features ──────────────────
    df['Qual_TotalSF']    = df['OverallQual'] * df['TotalSF']
    df['Qual_GrLivArea']  = df['OverallQual'] * df['GrLivArea']
    df['Qual_GarageArea'] = df['OverallQual'] * df['GarageArea'].fillna(0)
    df['Qual_TotalBath']  = df['OverallQual'] * df['TotalBath']

# ── Neighborhood target encoding ─────────────

neighbor_mean = train.groupby('Neighborhood')['SalePrice'].mean()

train['Neighborhood_Mean'] = train['Neighborhood'].map(neighbor_mean)
test['Neighborhood_Mean']  = test['Neighborhood'].map(neighbor_mean)

# Fill any test neighborhoods not seen in train
test['Neighborhood_Mean']  = test['Neighborhood_Mean'].fillna(
    neighbor_mean.mean()
)

print("Feature engineering done")
print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

Feature engineering done
Train shape: (1460, 96)
Test shape:  (1459, 95)


In [4]:
# ── 4. COMBINE train+test FOR CLEANING ───────────────
ntrain = train.shape[0]
all_data = pd.concat([train.drop('SalePrice', axis=1), test])
print(f"Combined shape: {all_data.shape}")

# None fills 
none_cols = ['PoolQC','MiscFeature','Alley','Fence',
             'FireplaceQu','GarageType','GarageFinish',
             'GarageQual','GarageCond','BsmtQual',
             'BsmtCond','BsmtExposure','BsmtFinType1',
             'BsmtFinType2','MasVnrType']
for col in none_cols:
    all_data[col] = all_data[col].fillna('None')

# Zero fills 
zero_cols = ['GarageYrBlt','GarageArea','GarageCars',
             'BsmtFinSF1','BsmtFinSF2','BsmtUnfSF',
             'TotalBsmtSF','BsmtFullBath','BsmtHalfBath',
             'MasVnrArea']
for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

# Median fills 
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage']\
                          .transform(lambda x: x.fillna(x.median()))

# Mode fills 
mode_cols = ['MSZoning','Electrical','KitchenQual',
             'Exterior1st','Exterior2nd','SaleType','Functional']
for col in mode_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

print(f"Missing remaining: {all_data.isnull().sum().sum()}")

# Drop Utilities from combined data
all_data = all_data.drop(columns=['Utilities'])
print(f"Missing remaining: {all_data.isnull().sum().sum()}")
print(f"Shape: {all_data.shape}")

Combined shape: (2919, 95)
Missing remaining: 2
Missing remaining: 0
Shape: (2919, 94)


In [5]:
# ── 5. DROP USELESS ──────────────────────────────────
all_data = all_data.drop(columns=['Street', 'Id'], errors='ignore')
print(f"Shape after dropping: {all_data.shape}")

# ── 6. ORDINAL ENCODING ──────────────────────────────
ordinal_maps = {
    'ExterQual':  {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'ExterCond':  {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'BsmtQual':   {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'BsmtCond':   {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'HeatingQC':  {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'KitchenQual':{'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'FireplaceQu':{'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'GarageQual': {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'GarageCond': {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'PoolQC':     {'None':0,'Fa':1,'TA':2,'Gd':3,'Ex':4},
    'BsmtExposure':{'None':0,'No':1,'Mn':2,'Av':3,'Gd':4},
    'BsmtFinType1':{'None':0,'Unf':1,'LwQ':2,'Rec':3,
                    'BLQ':4,'ALQ':5,'GLQ':6},
    'BsmtFinType2':{'None':0,'Unf':1,'LwQ':2,'Rec':3,
                    'BLQ':4,'ALQ':5,'GLQ':6},
    'GarageFinish':{'None':0,'Unf':1,'RFn':2,'Fin':3},
    'Fence':       {'None':0,'MnWw':1,'GdWo':2,'MnPrv':3,'GdPrv':4},
    'LandSlope':   {'Gtl':0,'Mod':1,'Sev':2},
    'LotShape':    {'Reg':0,'IR1':1,'IR2':2,'IR3':3},
    'PavedDrive':  {'N':0,'P':1,'Y':2},
    'CentralAir':  {'N':0,'Y':1},
    'Functional':  {'Sal':0,'Sev':1,'Maj2':2,'Maj1':3,
                    'Mod':4,'Min2':5,'Min1':6,'Typ':7},
}
for col, mapping in ordinal_maps.items():
    all_data[col] = all_data[col].map(mapping)

# ── 7. ONE-HOT ENCODE ────────────────────────────────
nominal_cols = all_data.select_dtypes(include=['object']).columns.tolist()
all_data = pd.get_dummies(all_data, columns=nominal_cols)
print(f"Final shape: {all_data.shape}")

# ── 8. SPLIT BACK ────────────────────────────────────
X_train = all_data[:ntrain]
X_test  = all_data[ntrain:]
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

Shape after dropping: (2919, 92)
Final shape: (2919, 236)
X_train: (1460, 236), X_test: (1459, 236)


In [6]:
# ── 9. TRAIN AND EVALUATE ────────────────────────────
models = {
    'XGBoost': XGBRegressor(
        n_estimators=3000,
        learning_rate=0.01,
        max_depth=3,
        subsample=0.7,
        colsample_bytree=0.7,
        gamma=0.01,
        reg_alpha=0.01,
        reg_lambda=1.0,
        random_state=42
    ),
    'LightGBM': LGBMRegressor(
        n_estimators=3000,
        learning_rate=0.01,
        max_depth=3,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=0.01,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1
    ),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train,
        cv=5,
        scoring='neg_mean_squared_error'
    )
    rmse = np.sqrt(-scores).mean()
    results[name] = rmse
    print(f"{name}: RMSE = {rmse:.4f}")

# ── 10. ENSEMBLE AND SUBMIT ──────────────────────────
for name, model in models.items():
    model.fit(X_train, y_train)

preds_xgb  = np.expm1(models['XGBoost'].predict(X_test))
preds_lgbm = np.expm1(models['LightGBM'].predict(X_test))

# Equal weight since both are similar
ensemble = 0.5 * preds_xgb + 0.5 * preds_lgbm

output = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': ensemble
})
output.to_csv('submission.csv', index=False)
print(f"\nMin:  ${ensemble.min():,.0f}")
print(f"Max:  ${ensemble.max():,.0f}")
print(f"Mean: ${ensemble.mean():,.0f}")
print("Done!")

XGBoost: RMSE = 0.1169
LightGBM: RMSE = 0.1233

Min:  $42,265
Max:  $543,664
Mean: $177,857
Done!
